# Interacting with Cassandra and Demonstrating its Wide-Column Nature

This notebook demonstrates how to interact with a Cassandra database using the Python `cassandra-driver`. We will highlight Cassandra's nature as a wide-column store, where rows in the same table can have different columns.

## 1. Installation

First, you need to install the `cassandra-driver` for Python. You can do this by running the following command in your terminal:

In [23]:
!pip install cassandra-driver


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## 2. Connecting to Cassandra

Now, let's establish a connection to the Cassandra instance. We assume Cassandra is running on `localhost` at port `9042`, as configured in the `docker-compose.yml` file.

In [33]:
from cassandra.cluster import Cluster

try:
    # Provide the contact points (IP addresses) of your Cassandra nodes.
    # In our Docker setup, it's just the local machine.
    cluster = Cluster(['127.0.0.1'], port=9042)

    # Connect to the cluster
    session = cluster.connect()
    print("Successfully connected to Cassandra!")
except Exception as e:
    print(f"Failed to connect to Cassandra: {e}")

Successfully connected to Cassandra!


## 3. Creating a Keyspace

A keyspace in Cassandra is a namespace that defines how data is replicated on nodes. It's similar to a schema or database in a relational database.

In [34]:
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS user_profiles
        WITH replication = { 'class': 'SimpleStrategy', 'replication_factor': '1' }
    """)
    print("Keyspace 'user_profiles' created successfully or already exists.")
except Exception as e:
    print(f"Error creating keyspace: {e}")

Keyspace 'user_profiles' created successfully or already exists.


Let's switch to our new keyspace:

In [35]:
try:
    session.set_keyspace('user_profiles')
    print("Switched to keyspace 'user_profiles'.")
except Exception as e:
    print(f"Error switching keyspace: {e}")

Switched to keyspace 'user_profiles'.


## 4. Creating a Table and Demonstrating the Wide-Column Model

Now, we'll create a table to store user data. In Cassandra, you define the primary key, which consists of a partition key and optional clustering columns. All other columns are not fixed, which allows for the wide-column feature.

Our table `users` will have a `user_id` as the partition key. We will pre-define some common columns like `username` and `email`, but we can add more columns on the fly for different rows.

In [36]:
try:
    session.execute("""
    CREATE TABLE IF NOT EXISTS users (
        user_id UUID PRIMARY KEY,
        username text,
        email text
    )
    """)
    print("Table 'users' created successfully or already exists.")
except Exception as e:
    print(f"Error creating table: {e}")

Table 'users' created successfully or already exists.


### Inserting Data: The Wide-Column Feature in Action

Let's insert two users. The first user will have the basic information. The second user will have additional, dynamically added attributes: `city` and `interests`.

In [37]:
import uuid

# User 1: Basic Info
user1_id = uuid.uuid4()
try:
    # Using prepared statements is the recommended way to pass parameters.
    query = "INSERT INTO users (user_id, username, email) VALUES (?, ?, ?)"
    prepared = session.prepare(query)
    session.execute(prepared, (user1_id, 'alice', 'alice@example.com'))
    print(f"Inserted basic info for user 'alice' with ID {user1_id}")
except Exception as e:
    print(f"Error inserting data for alice: {e}")

# User 2: Adding more columns dynamically
# We need to alter the table to add the new columns before we can insert data into them.
try:
    session.execute("ALTER TABLE users ADD city text")
    session.execute("ALTER TABLE users ADD interests list<text>")
    print("Altered table to add 'city' and 'interests' columns.")
except Exception as e:
    # This will likely fail on subsequent runs if the columns already exist, which is fine.
    print(f"Could not alter table (maybe columns already exist?): {e}")

user2_id = uuid.uuid4()
try:
    query = "INSERT INTO users (user_id, username, email, city, interests) VALUES (?, ?, ?, ?, ?)"
    prepared = session.prepare(query)
    session.execute(prepared, (user2_id, 'bob', 'bob@example.com', 'New York', ['hiking', 'reading']))
    print(f"Inserted extended info for user 'bob' with ID {user2_id}")
except Exception as e:
    print(f"Error inserting data for bob: {e}")

Inserted basic info for user 'alice' with ID ccd3c5dc-1a32-422a-98d0-39d95cc5e404
Altered table to add 'city' and 'interests' columns.
Inserted extended info for user 'bob' with ID f6c4905a-3206-46d7-a5c1-190a31e8a7cb


## 5. Querying the Data

Now, let's retrieve the data for both users. Notice how Cassandra returns `null` for the columns that are not set for a particular row. This demonstrates the schema-on-read flexibility.

In [38]:
try:
    rows = session.execute("SELECT * FROM users")
    print("Querying all users:")
    for row in rows:
        print(f"User ID: {row.user_id}")
        print(f"  Username: {row.username}")
        print(f"  Email: {row.email}")
        # Check if the column exists for the row before accessing it
        if 'city' in row and row.city:
            print(f"  City: {row.city}")
        if 'interests' in row and row.interests:
            print(f"  Interests: {row.interests}")
        print("---")
except Exception as e:
    print(f"Error querying data: {e}")

Querying all users:
User ID: f6c4905a-3206-46d7-a5c1-190a31e8a7cb
  Username: bob
  Email: bob@example.com
---
User ID: ccd3c5dc-1a32-422a-98d0-39d95cc5e404
  Username: alice
  Email: alice@example.com
---


## 6. Cleaning Up

Finally, let's close the connection to the Cassandra cluster.

In [39]:
try:
    cluster.shutdown()
    print("Connection to Cassandra closed.")
except Exception as e:
    print(f"Error closing connection: {e}")

Connection to Cassandra closed.
